# Clase 10 — Feature Engineering
## Feature Engineering Challenge — Dataset: `clientes_features.csv`

**Regla del reto:** todos usan `LinearRegression`. No se puede cambiar el
algoritmo — solo se puede modificar `X`.

**Objetivo:** predecir `gasto_siguiente_mes` y reducir el MAE.

> Todo el código está comentado a propósito. Ve descomentando celda por
> celda conforme los equipos avanzan en el reto.

## Paso 0 — Configuración inicial

In [ ]:
# import pandas as pd
# import numpy as np

# from sklearn.model_selection import train_test_split
# from sklearn.linear_model import LinearRegression
# from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score


## Cargar los datos

In [ ]:
# df = pd.read_csv("../data/clientes_features.csv")
# print(df.head())
# df.info()


## Target
`y = gasto_siguiente_mes`

In [ ]:
# y = df["gasto_siguiente_mes"]


---
## Fase 1 — Modelo base
Usar solamente: `edad`, `ingreso`, `visitas`, `compras`, `gasto_total`.

In [ ]:
# columnas_base = ["edad", "ingreso", "visitas", "compras", "gasto_total"]

# X_base = df[columnas_base]

# X_train_base, X_test_base, y_train, y_test = train_test_split(
    # X_base, y, test_size=0.20, random_state=42
# )

# modelo_base = LinearRegression()
# modelo_base.fit(X_train_base, y_train)

# pred_base = modelo_base.predict(X_test_base)

# mae_base = mean_absolute_error(y_test, pred_base)
# rmse_base = mean_squared_error(y_test, pred_base) ** 0.5
# r2_base = r2_score(y_test, pred_base)

# print("MAE_base:", mae_base, " RMSE_base:", rmse_base, " R2_base:", r2_base)


---
## Fase 2 — Crear al menos 4 tipos de features

1. una variable ratio
2. una variable temporal
3. una transformación
4. una variable categórica codificada

### Validar antes de crear ratios
Este dataset tiene, a propósito, algunas filas con `compras = 0`.

In [ ]:
# print("Filas con compras = 0:", (df["compras"] == 0).sum())


### 1) Ratios

In [ ]:
# compras_seguras = df["compras"].replace(0, 1)  # evita división entre cero

# df["gasto_por_compra"] = df["gasto_total"] / compras_seguras
# df["visitas_por_compra"] = df["visitas"] / compras_seguras


### 2) Variable temporal — antigüedad del cliente

In [ ]:
# df["fecha_registro"] = pd.to_datetime(df["fecha_registro"])

# fecha_referencia = pd.Timestamp("2026-08-21")
# df["antiguedad_cliente"] = (fecha_referencia - df["fecha_registro"]).dt.days


### 3) Transformación — log de ingreso

In [ ]:
# df["ingreso_log"] = np.log1p(df["ingreso"])


### 4) Variable categórica — ciudad (One-Hot Encoding)

In [ ]:
# from sklearn.compose import ColumnTransformer
# from sklearn.preprocessing import StandardScaler, OneHotEncoder

# numericas = [
    # "edad",
    # "ingreso_log",
    # "visitas",
    # "compras",
    # "gasto_total",
    # "antiguedad_cliente",
    # "gasto_por_compra",
    # "visitas_por_compra"
# ]

# categoricas = ["ciudad"]


---
## Fase 3 — Pipeline con StandardScaler + OneHotEncoder + LinearRegression

In [ ]:
# preprocesador = ColumnTransformer(
    # transformers=[
        # ("num", StandardScaler(), numericas),
        # ("cat", OneHotEncoder(handle_unknown="ignore"), categoricas)
    # ]
# )


In [ ]:
# from sklearn.pipeline import Pipeline

# pipeline_features = Pipeline(
    # steps=[
        # ("prep", preprocesador),
        # ("modelo", LinearRegression())
    # ]
# )

# X_features = df[numericas + categoricas]

# X_train_feat, X_test_feat, y_train, y_test = train_test_split(
    # X_features, y, test_size=0.20, random_state=42
# )

# pipeline_features.fit(X_train_feat, y_train)

# pred_features = pipeline_features.predict(X_test_feat)


---
## Fase 4 — Comparar MAE_base vs MAE_features

In [ ]:
# mae_features = mean_absolute_error(y_test, pred_features)
# rmse_features = mean_squared_error(y_test, pred_features) ** 0.5
# r2_features = r2_score(y_test, pred_features)

# mejora = mae_base - mae_features

# print("MAE_base:      ", mae_base, " R2_base:", r2_base)
# print("MAE_features:  ", mae_features, " R2_features:", r2_features)
# print("Mejora (MAE):  ", mejora)


---
## Fase 5 — Discusión en equipo

Respondan antes de revisar la siguiente celda:

1. ¿Qué feature nueva ayudó más?
2. ¿Qué feature no ayudó?
3. ¿Alguna produjo overfitting?
4. ¿Existe riesgo de leakage en alguna de las variables creadas?
5. ¿Qué variables conservarían para un modelo en producción?

### Pista para el análisis por feature (opcional)
Pueden comparar el modelo agregando una feature nueva a la vez sobre el
modelo base, para ver la contribución individual de cada una.

In [ ]:
# def evaluar_con_extra(col_extra, es_categorica=False):
    # cols_num = columnas_base.copy()
    # cols_cat = []
    # if es_categorica:
        # cols_cat = [col_extra]
    # else:
        # cols_num = cols_num + [col_extra]

    # X_try = df[cols_num + cols_cat]
    # X_tr, X_te, y_tr, y_te = train_test_split(X_try, y, test_size=0.20, random_state=42)

    # prep_try = ColumnTransformer(
        # transformers=[
            # ("num", StandardScaler(), cols_num),
            # ("cat", OneHotEncoder(handle_unknown="ignore"), cols_cat)
        # ]
    # )
    # pipe_try = Pipeline([("prep", prep_try), ("modelo", LinearRegression())])
    # pipe_try.fit(X_tr, y_tr)
    # pred_try = pipe_try.predict(X_te)

    # return mean_absolute_error(y_te, pred_try)


# # for col in ["gasto_por_compra", "visitas_por_compra", "antiguedad_cliente", "ingreso_log"]:
# #     print(col, "->", evaluar_con_extra(col))
# # print("ciudad ->", evaluar_con_extra("ciudad", es_categorica=True))
